In [1]:
!pip install -U langchain langchain-google-genai langchain-community langchain-neo4j

In [1]:
!pip install neo4j pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 3.7 MB/s eta 0:00:00


In [5]:
import pandas as pd
from neo4j import GraphDatabase

# --- 1. KONFIGURASI KONEKSI ---
URI = "bolt://44.204.196.179:7687"
USER = "neo4j"
PASSWORD = "berth-habits-pronouns"

# Tes Koneksi
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
try:
    driver.verify_connectivity()
    print("✅ Koneksi ke Neo4j Sandbox Berhasil!")
except Exception as e:
    print("❌ Koneksi Gagal:", e)

# --- 2. MEMBACA & MERAPIKAN DATA CSV ---
df = pd.read_csv('integrated_game_data (1).csv')

# Memecah teks yang dipisah koma menjadi list
def split_and_trim(text):
    if pd.isna(text): return []
    return [item.strip() for item in str(text).split(',')]

# Terapkan fungsi ke kolom yang berisi banyak nilai
df['developers'] = df['developers'].apply(split_and_trim)
df['genres'] = df['genres'].apply(split_and_trim)
df['platforms'] = df['platforms'].apply(split_and_trim)

# Ubah format data
data_to_ingest = df.to_dict('records')
print(f"✅ Data CSV siap: {len(data_to_ingest)} games.")

# --- 3. FUNGSI INGESTI ---
def create_graph_data(tx, games_data):
    query = """
    UNWIND $games AS game

    // 1. Buat Node Game (Hanya Title)
    MERGE (g:Game {title: game.title})

    // 2. Buat Node Developer & Relasinya
    WITH g, game
    UNWIND game.developers AS dev_name
    MERGE (d:Developer {name: dev_name})
    MERGE (g)-[:DEVELOPED_BY]->(d)

    // 3. Buat Node Genre & Relasinya
    WITH g, game
    UNWIND game.genres AS genre_name
    MERGE (cat:Genre {name: genre_name})
    MERGE (g)-[:HAS_GENRE]->(cat)

    // 4. Buat Node Platform & Relasinya
    WITH g, game
    UNWIND game.platforms AS platform_name
    MERGE (p:Platform {name: platform_name})
    MERGE (g)-[:AVAILABLE_ON]->(p)
    """
    tx.run(query, games=games_data)

# --- 4. EKSEKUSI KE DATABASE ---
print("Memulai ingesti data ke Neo4j...")
with driver.session() as session:
    session.execute_write(create_graph_data, data_to_ingest)

print("TAHAP 1 done, Data baru berhasil dimasukkan ke Neo4j Sandbox.")

✅ Koneksi ke Neo4j Sandbox Berhasil!
✅ Data CSV siap: 298 games.
Memulai ingesti data ke Neo4j...
🎉 TAHAP 1 SELESAI! Data baru berhasil dimasukkan ke Neo4j Sandbox.


In [7]:
# --- TAHAP 2: GRAPH ANALYTICS & MACHINE LEARNING (GDS) ---

def run_gds_algorithms(tx):
    print("1. Membuat In-Memory Graph untuk GDS...")
    # Menghapus graph 'gameGraph' jika sebelumnya sudah ada
    tx.run("CALL gds.graph.drop('gameGraph', false) YIELD graphName;")

    # Memproyeksikan data ke memori GDS Undirected agar relasi bisa dibaca bolak-balik
    tx.run("""
    CALL gds.graph.project(
      'gameGraph',
      ['Game', 'Developer', 'Genre', 'Platform'],
      {
          DEVELOPED_BY: {orientation: 'UNDIRECTED'},
          HAS_GENRE: {orientation: 'UNDIRECTED'},
          AVAILABLE_ON: {orientation: 'UNDIRECTED'}
      }
    )
    """)
    print("✅ In-Memory Graph 'gameGraph' berhasil dibuat!")

    print("\n2. Menjalankan Graph Analytics: Degree Centrality (Top 5 Paling Populer)")
    # Mencari Node yang paling banyak relasinya
    analytics_result = tx.run("""
    CALL gds.degree.stream('gameGraph')
    YIELD nodeId, score
    WITH gds.util.asNode(nodeId) AS n, score
    WHERE NOT 'Game' IN labels(n) // <-- Ini sintaks Cypher yang benar
    RETURN coalesce(n.name, n.title) AS Nama, labels(n)[0] AS Kategori, score AS Total_Koneksi
    ORDER BY Total_Koneksi DESC LIMIT 5
    """)
    df_analytics = pd.DataFrame([r.values() for r in analytics_result], columns=["Nama", "Kategori", "Total Koneksi"])
    print(df_analytics.to_string(index=False))

    print("\n3. Menjalankan Machine Learning: FastRP Embedding & KNN (Sistem Rekomendasi Game)")
    # Step A: Membuat Embedding (Representasi Vektor/Angka dari struktur Graf)
    tx.run("""
    CALL gds.fastRP.mutate('gameGraph', {
      embeddingDimension: 64,
      randomSeed: 42,
      mutateProperty: 'embedding'
    })
    """)

    # Step B: Mencari kemiripan antar Game menggunakan K-Nearest Neighbors (KNN)
    ml_result = tx.run("""
    CALL gds.knn.stream('gameGraph', {
        nodeLabels: ['Game'],
        nodeProperties: ['embedding'],
        topK: 1,
        randomSeed: 42,
        concurrency: 1,
        sampleRate: 1.0,
        deltaThreshold: 0.0
    })
    YIELD node1, node2, similarity
    RETURN gds.util.asNode(node1).title AS Game_Asal,
           gds.util.asNode(node2).title AS Rekomendasi_Game,
           similarity AS Skor_Kemiripan
    ORDER BY Skor_Kemiripan DESC LIMIT 5
    """)
    df_ml = pd.DataFrame([r.values() for r in ml_result], columns=["Game", "Rekomendasi", "Skor Kemiripan"])
    print(df_ml.to_string(index=False))

# Eksekusi fungsi GDS
with driver.session() as session:
    session.execute_write(run_gds_algorithms)

print("TAHAP 2 done, tier 1 & 2 done!.")

1. Membuat In-Memory Graph untuk GDS...
✅ In-Memory Graph 'gameGraph' berhasil dibuat!

2. Menjalankan Graph Analytics: Degree Centrality (Top 5 Paling Populer)
             Nama  Kategori  Total Koneksi
Microsoft Windows  Platform           94.0
          Unknown Developer           61.0
          Unknown  Platform           45.0
      ZX Spectrum  Platform           40.0
     Commodore 64  Platform           40.0

3. Menjalankan Machine Learning: FastRP Embedding & KNN (Sistem Rekomendasi Game)
                      Game                Rekomendasi  Skor Kemiripan
Battleground 2: Gettysburg   Battleground 5: Antietam             1.0
      AFL Premiership 2006         Ninja Gaiden Sigma             1.0
                  AR Games                     Jumbly             1.0
       ARCA Sim Racing '08                   AR Games             1.0
  Battleground 5: Antietam Battleground 2: Gettysburg             1.0

🎉 TAHAP 2 SELESAI! Syarat Tier 1 & Tier 2 Terpenuhi.


In [9]:

!pip install --upgrade langchain langchain-google-genai langchain-community langchain-neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.6
    Uninstalling langchain-1.3.6:
      Successfully uninstalled langchain-1.3.6


In [1]:
# CELL 5: INSTALASI
!pip install --upgrade langchain langchain-google-genai langchain-community langchain-neo4j

In [8]:
# --- TAHAP 3: TEXT TO CYPHER (AUTO-DETECT MODEL & UPDATE) ---

# 1. Update library inti
!pip install -q -U google-generativeai langchain-google-genai

import os
import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

# 2. SETUP KREDENSIAL
os.environ["GOOGLE_API_KEY"] = "AQ.Ab8RN6J0-GhfRsTrmJpNhUt3pdPRk7AHgGofJ7TidwSkz4mPJg"

URI = "bolt://44.204.196.179:7687"
USER = "neo4j"
PASSWORD = "berth-habits-pronouns"

# 3. AUTO-DETEKSI MODEL GEMINI
print("Mencari model AI yang tersedia di server...")
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Mengambil daftar model yang aktif dan mendukung Text-Generation
tersedia = [m.name.replace('models/', '') for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

# AI amemilih otomatis mana model yang aktif (Prioritas: 1.5-flash / 1.5-pro)
model_pilihan = tersedia[0] if tersedia else "gemini-1.5-flash"
for m in tersedia:
    if "1.5-flash" in m:
        model_pilihan = m
        break
    elif "1.5-pro" in m:
        model_pilihan = m

print(f"✅ Berhasil! Model yang akan digunakan: {model_pilihan}\n")

# 4. KONEKSI & PIPELINE AI
print("⏳ Menghubungkan ke Neo4j...")
try:
    graph = Neo4jGraph(url=URI, username=USER, password=PASSWORD)
    print("✅ Berhasil membaca skema Neo4j!")
except Exception as e:
    print(f"❌ Gagal konek ke Neo4j: {e}")

llm = ChatGoogleGenerativeAI(model=model_pilihan, temperature=0)

try:
    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True, # Agar Cypher Query muncul di layar
        allow_dangerous_requests=True
    )
    print("✅ Pipeline AI Text-to-Cypher 100% Siap!\n")
    print("=" * 50)
except Exception as e:
    print(f"❌ Error saat membuat pipeline: {e}")

# 5. UJI COBA BERTANYA
pertanyaan_1 = "Ada berapa banyak judul game di dalam database ini?"
print(f"User: {pertanyaan_1}")
try:
    jawaban_1 = chain.invoke({"query": pertanyaan_1})
    print(f"Jawaban AI: {jawaban_1['result']}")
except Exception as e:
    print(f"Error AI: {e}")

print("-" * 50)

pertanyaan_2 = "Sebutkan 3 judul game yang memiliki genre action game!"
print(f"User: {pertanyaan_2}")
try:
    jawaban_2 = chain.invoke({"query": pertanyaan_2})
    print(f"Jawaban AI: {jawaban_2['result']}")
except Exception as e:
    print(f"Error AI: {e}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


⏳ Mencari model AI yang tersedia di server...
✅ Berhasil! Model yang akan digunakan: gemini-2.5-flash

⏳ Menghubungkan ke Neo4j...
✅ Berhasil membaca skema Neo4j!
✅ Pipeline AI Text-to-Cypher 100% Siap!

User: Ada berapa banyak judul game di dalam database ini?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (g:Game) RETURN count(g)
Full Context:
[{'count(g)': 298}]

> Finished chain.
🤖 Jawaban AI: Ada 298 judul game di dalam database ini.
--------------------------------------------------
User: Sebutkan 3 judul game yang memiliki genre action game!


> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (game:Game)-[:HAS_GENRE]->(genre:Genre)
WHERE genre.name = 'action game'
RETURN game.title
LIMIT 3

Full Context:
[{'game.title': '.kkrieger'}, {'game.title': '720°'}, {'game.title': 'A View to a Kill'}]

> Finished chain.
🤖 Jawaban AI: .kkrieger, 720°, A View to a Kill adalah 3 judul game yang memiliki genre action game.


In [10]:
# --- TAHAP 4: LLM GRAPH BUILDER (SYARAT TIER 4 FINAL) ---

# 1. Instalasi library khusus untuk ekstraksi Graph dari teks
!pip install -q langchain-experimental

from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

print("Menyiapkan AI Graph Builder...")

# 2. Inisialisasi Transformer menggunakan LLM yang sudah aktif
llm_transformer = LLMGraphTransformer(llm=llm)

# 3. Beri AI sebuah Teks Berita / Artikel Bebas (Unstructured Data)
teks_artikel = """
Genshin Impact adalah action role-playing game yang sangat populer.
Game ini dikembangkan secara eksklusif oleh miHoYo.
Genshin Impact tersedia dan dapat dimainkan di platform PlayStation 5, iOS, dan Android.
"""
print("Membaca teks artikel berikut:")
print(f'"{teks_artikel.strip()}"\n')

# 4. Proses Ekstraksi AI (Merubah Teks menjadi Graph)
print("AI sedang berpikir: Mengekstrak Entitas & Relasi dari kalimat...")
documents = [Document(page_content=teks_artikel)]

try:
    graph_documents = llm_transformer.convert_to_graph_documents(documents)

    print("✅ Ekstraksi berhasil! AI menemukan data berikut:")
    print("NODES (Entitas):")
    for node in graph_documents[0].nodes:
        print(f" 🟢 {node.id} (Tipe: {node.type})")

    print("\nRELATIONSHIPS (Relasi):")
    for rel in graph_documents[0].relationships:
        print(f" 🔗 {rel.source.id} --[{rel.type}]--> {rel.target.id}")

    # 5. Memasukkan data baru ini ke Neo4j
    print("\n⏳ Menyuntikkan pengetahuan baru ini ke dalam Neo4j...")
    graph.add_graph_documents(graph_documents)
    print("Data unstructured berhasil menjadi Graph. Tier 4 done")

except Exception as e:
    print(f"❌ Terjadi kesalahan saat ekstraksi: {e}")

⏳ Menyiapkan AI Graph Builder...
📄 Membaca teks artikel berikut:
"Genshin Impact adalah action role-playing game yang sangat populer. 
Game ini dikembangkan secara eksklusif oleh miHoYo. 
Genshin Impact tersedia dan dapat dimainkan di platform PlayStation 5, iOS, dan Android."

🤖 AI sedang berpikir: Mengekstrak Entitas & Relasi dari kalimat...
✅ Ekstraksi berhasil! AI menemukan data berikut:
NODES (Entitas):
 🟢 Genshin Impact (Tipe: Game)
 🟢 Action Role-Playing Game (Tipe: Genre)
 🟢 Mihoyo (Tipe: Developer)
 🟢 Playstation 5 (Tipe: Platform)
 🟢 Ios (Tipe: Platform)
 🟢 Android (Tipe: Platform)

RELATIONSHIPS (Relasi):
 🔗 Genshin Impact --[IS_A]--> Action Role-Playing Game
 🔗 Genshin Impact --[DEVELOPED_BY]--> Mihoyo
 🔗 Genshin Impact --[AVAILABLE_ON]--> Playstation 5
 🔗 Genshin Impact --[AVAILABLE_ON]--> Ios
 🔗 Genshin Impact --[AVAILABLE_ON]--> Android

⏳ Menyuntikkan pengetahuan baru ini ke dalam Neo4j...
Data unstructured berhasil menjadi Graph. Tier 4 done
